# Scraping experiments

This notebook holds experiments conducted to try and gather extra data about racquets from the Tennis Warehouse University website. Many of their tools reference extra data about racquets like sweet spot scores, twist weight, vibration, etc. It might be useful to gather this extra information for each of the 320 racquets in the cleaned data to improve the "intelligence" of the search. For now, this notebook just explores getting the mapping between racquet name and API code and tries to hit the API endpoint once to see if I can successfully pull data.

In [ ]:
from pprint import pprint
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json


In [ ]:
# Get webpage contents 
webpage = requests.get("https://twu.tennis-warehouse.com/cgi-bin/compareracquets.cgi")
soup = BeautifulSoup(webpage.content, "html.parser")

In [33]:
# Isolate top drop-down and get all the option tags
select_tag = soup.find("select", {"name": "racquetA"})
options = select_tag.find_all("option")

# For each option tag (after first one), get code after racquet prefix 
# and the name of the racquet
racquet_codes = [
    {
        "code": opt.get("value").replace("racquet", ""),
        "name": opt.text.strip()
    }
    for opt in options
    if opt.get("value") != "none"
    
]

In [ ]:
racquet_codes[:10] # First 10 racquets

[{'code': 'A-ABR', 'name': 'Adidas Barricade'},
 {'code': 'A-ABTR', 'name': 'Adidas Barricade Tour'},
 {'code': 'A-AFR', 'name': 'Adidas Feather'},
 {'code': 'A-ARR', 'name': 'Adidas Response'},
 {'code': 'A-A109', 'name': 'Asics Asics 109'},
 {'code': 'A-A116', 'name': 'Asics Asics 116'},
 {'code': 'A-A125', 'name': 'Asics Asics 125'},
 {'code': 'A-ABZ100', 'name': 'Asics BZ 100'},
 {'code': 'A-AM3CON', 'name': 'Avery M3 Control'},
 {'code': 'A-AM3POW', 'name': 'Avery M3 Power'}]

In [38]:
len(racquet_codes)

1355

`racquet_codes` contains **every** racquet that TW has ever analyzed (which is much more than the amount they currently list for sale). So, I need some way to filter out so that it's only the ones with the names in my scraped racquet dataset.

In [46]:
# Import raw scraped racquet data
racquet_listings = pd.read_csv("../data/raw/racquets_raw_02_14_26.csv")
racquet_names = set(racquet_listings["racquet_name"])

filtered_racquet_codes = [
    {
        "name": entry["name"],
        "code": entry["code"]
    }
    for entry in racquet_codes
    if entry["name"] in racquet_names
]

len(filtered_racquet_codes), len(racquet_names)

(122, 379)

Unfortunately, there appear to be very severe naming discrepancies between the listed racquets and TWU. That or most of the listed racquets haven't been inputted into the data behind TWU yet. While exploring this avenue is interesting and could yield some more informative data about each racquet, it seems to be out of scope for the current version of the progress. 

**So, I'm leaving this notebook here as a general showcase of a potential future avenue to explore.**

In [ ]:
# Test a simple POST request to get "comparison tool" response for Babolat Pure Aero 2023
url = "https://twu.tennis-warehouse.com/cgi-bin/compareracquetsdata.cgi"
racquet_param = "?racquet" + filtered_racquet_codes[0]["code"]
response = requests.post(url = url + racquet_param)
response.raise_for_status()

In [ ]:
# Response JSON
response.json()

{'pcode': 'BARO',
 'mfg': 'Babolat',
 'racquet': 'Pure Aero 2023',
 'headsize': 100,
 'length': 27.0,
 'weight': 318,
 'balance': 33.0,
 'swingweight': 322,
 'flex': 65,
 'acor': 41,
 'sweet': 17,
 'rccode': 'RCBABOLAT',
 'current': 'x',
 'twistweight': 14.6,
 'vibration': '141',
 'units': '',
 'changed': 'racquetA'}

In [ ]:
# Response copied from Inspect Element - looks the same but diff order
pprint({"pcode":"BARO","mfg":"Babolat","racquet":"Pure Aero 2023","headsize":100,"length":27.00,"weight":318,"balance":33.0,"swingweight":322,"flex":65,"acor":41,"sweet":17,"rccode":"RCBABOLAT","current":"x","twistweight":14.6,"vibration":"141","units":"","changed":"racquetA"})


{'acor': 41,
 'balance': 33.0,
 'changed': 'racquetA',
 'current': 'x',
 'flex': 65,
 'headsize': 100,
 'length': 27.0,
 'mfg': 'Babolat',
 'pcode': 'BARO',
 'racquet': 'Pure Aero 2023',
 'rccode': 'RCBABOLAT',
 'sweet': 17,
 'swingweight': 322,
 'twistweight': 14.6,
 'units': '',
 'vibration': '141',
 'weight': 318}


In [77]:
len(response.json()), len({"pcode":"BARO","mfg":"Babolat","racquet":"Pure Aero 2023","headsize":100,"length":27.00,"weight":318,"balance":33.0,"swingweight":322,"flex":65,"acor":41,"sweet":17,"rccode":"RCBABOLAT","current":"x","twistweight":14.6,"vibration":"141","units":"","changed":"racquetA"})

(17, 17)